#Перевірка статиcтичних гіпотез. Z-test.



0. Зчитайте дані з `data.csv` у змінну data, яка має тип pandas.DataFrame. Ми будемо далі працювати з цією змінною.

In [13]:
import pandas as pd
import numpy as np
import scipy as stats
from statsmodels.stats import weightstats as stests

from google.colab import files

uploaded = files.upload()
data = pd.read_csv("data.csv")
data.head()

Saving data.csv to data (3).csv


,Unnamed: 0,data
0,0,17.499453
1,1,19.662399
2,2,7.182823
3,3,29.841625
4,4,9.239386


Запустіть код нижче. Для коректної робити всіх подальших методів дані мають бути у вигляді одновимірного `numpy.ndarray` вектора та мати розмірність (100,). Така розмірність означає, що дані - одновимірні (колонка), якщо б розмірність була (100, 1), то дані сприймаються методами як двовимірні (таблиця), хоча для нас виглядати можуть так само.

In [15]:
data = data.data.values
data.shape

(100,)

**Завдання 1**. (7 балів)
Зчитані дані - це сума покупок в доларах наших покупців на сайті протягом липня 2025 року.

До цього місяця, ми мали продажі в середньому на чек $20.

Необхідно зʼясувати, чи вийшло у нас статистично значущо **підвищити** середній чек за липень 2025?

Аби дати відповідь - ми проведемо z-test на рівні значущості $\alpha=0.05$ двома способами. В цьому завданні ми будемо виконувати обчислення "вручну" з використанням бібліотек numpy та scipy.stats подібно до прикладу в лекції. Для цього виконайте наступні 7 кроків. Правильне виконання кожного з кроків оцінюється в 1 бал.

1. Запишіть параметри задачі у змінні Python:
    - `sample_mean` - $\bar{x}$, середнє значення в вибірці
    - `population_mean` - $\mu_0$, середнє значення в популяції (тобто те, з яким ми порівнюємо середнє вибіркове значення)
    - `population_std` - $\sigma$, вибіркове стандартне відхилення, яке ми вважаємо, що є рівним ст. відх. популяції, адже маємо достатньо велику вибірку.
    - `sample_size` - $n$, розмір вибірки
    - `alpha` - $\alpha$ рівень значущості
    
    **Увага!** Для обчислення стандартного відхилення маємо скористатись функцією `np.std(your_dataframe, ddof=1)`. Чому так - розбираємо в лекції про t-test.


2. Визначте для цієї задачі:
    - якою є гіпотеза $H_0$
    - якою є альтернативна гіпотеза $H_a$
    - з яким типом тесту ми маємо справу - лівосторонній, правосторонній чи двосторонній.

3. Обчисліть стандартну помилку SE.
4. Розрахуйте z-статистику.
5. Знайдіть критичне z-значення з допомогою бібліотеки stats.
6. Обчисліть p-value з допомогою бібліотеки stats.
7. Прийміть рішення, чи відхиляєте ви гіпотезу $H_0$. Для прийняття рішення зробіть і порівняння z-статистики з критичним значенням, і проаналізуйте p-value.

In [17]:
sample_mean = np.mean(data)
population_mean = 20
population_std = np.std(data, ddof=1)
sample_size = len(data)
alpha = 0.05

print(f"sample_mean     = {sample_mean:.4f}")
print(f"population_mean = {population_mean}")
print(f"population_std  = {population_std:.4f}")
print(f"sample_size     = {sample_size}")
print(f"alpha           = {alpha}")

sample_mean     = 19.3776
population_mean = 20
population_std  = 6.2536
sample_size     = 100
alpha           = 0.05


Гіпотези

H₀: μ = 20 (середній чек не змінився)
H₁: μ > 20 (середній чек збільшився)
Тип тесту: правосторонній (one-tailed)

In [20]:
standard_error = population_std / np.sqrt(sample_size)
print(f"standard_error  = {standard_error:.4f}")

standard_error  = 0.6254


In [23]:
z_statistic = (sample_mean - population_mean) / standard_error
print(f"z = {z_statistic:.3f}")

z = -0.995


In [28]:
#критичне значення
from scipy import stats

z_critical = stats.norm.ppf(1 - alpha)
print(f"z = {z_critical:.3f}")

z = 1.645


In [29]:
#p-value
p_value = 1 - stats.norm.cdf(z_statistic)
print(f"p-value = {p_value:.6f}")

p-value = 0.840216


Значення z-статистики (-0.995) є меншим за z-критичне (1.645), тобто не потрапляє в область відхилення Z ≥ 1,645, тому H0 не відхиляю. До того ж, p-value (0.84) є значно більшим за значення альфа (0.05). На рівні значущості 5% немає достатньо підстав стверджувати, що середній чек зріс в порівнянні з 20$ за липень 2025 року

**Завдання 2.** (3 бали)
Виконайте обчислення z-test з використанням бібліотеки statsmodels.

Отримайте z-статистику та р-значення.

Виведіть p-значення та зробіть висновок, чи ми маємо достатньо доказів, аби стверджувати, що середній чек зріс.

Чи зійшлись значення z-статистику та р-значення в цьому завданні з попередніми обчисленнями?

In [31]:
from statsmodels.stats import weightstats as stests

print("\n" + "=" * 60)
print("ВИКОНАННЯ Z-ТЕСТУ:")
print("=" * 60)

z_stat_sm, p_val_sm = stests.ztest(data,
                                  value = population_mean,
                                  alternative = 'larger')
print(f"Результати z-тесту:")
print(f"  Z-статистика: {z_stat_sm:.3f}")
print(f"  P-value: {p_val_sm:.6f}")
print()


ВИКОНАННЯ Z-ТЕСТУ:
Результати z-тесту:
  Z-статистика: -0.995
  P-value: 0.840216



Значення p-value та z-статистики зійшлися з попереднім розрахунком

In [33]:
# Інтерпретація результатів
print("ІНТЕРПРЕТАЦІЯ:")
print("-" * 40)
if p_val_sm < alpha:
    print(f"✅ ВІДХИЛЯЄМО H₀")
    print(f"   p-value ({p_val_sm:.6f}) < α ({alpha})")
    print("   Висновок: Середній чек статистично значуще виріс у липні 2025")
else:
    print(f"❌ НЕ ВІДХИЛЯЄМО H₀")
    print(f"   p-value ({p_val_sm:.6f}) ≥ α ({alpha})")
    print("   Висновок: Немає достатніх доказів про зростання середнього чеку у липні 2025")

# Порівняння з ручними розрахунками
print("\n" + "=" * 60)
print("ПОРІВНЯННЯ З РУЧНИМИ РОЗРАХУНКАМИ:")
print("=" * 60)
print(f"Z-статистика:")
print(f"  Ручний розрахунок: {z_statistic:.3f}")
print(f"  Statsmodels:       {z_stat_sm:.3f}")
print(f"  Різниця:           {abs(z_statistic - z_stat_sm):.3f}")
print()


ІНТЕРПРЕТАЦІЯ:
----------------------------------------
❌ НЕ ВІДХИЛЯЄМО H₀
   p-value (0.840216) ≥ α (0.05)
   Висновок: Немає достатніх доказів про зростання середнього чеку у липні 2025

ПОРІВНЯННЯ З РУЧНИМИ РОЗРАХУНКАМИ:
Z-статистика:
  Ручний розрахунок: -0.995
  Statsmodels:       -0.995
  Різниця:           0.000

